# 0.3 — Sampling parameters: what does a "distribution over responses" look like?

**Goal.** Same prompt, vary `temperature` and `top_p`, draw ~10 samples each, and build intuition for
how diverse the base model's response distribution actually is. Phase 1 treats "the assistant" as a
distribution $P_0(a \mid q)$; this is what that distribution looks like when you draw from it.

Two places to look:
1. **The first token.** The whole response distribution starts with a distribution over the first
   token. We can print it exactly (top-k probabilities) and see how temperature and top-p reshape it.
2. **Whole responses.** How many of 10 samples are unique? How long are they? How often does the model
   choose to stop?

In [1]:
import os, sys, time, json, textwrap, collections
from pathlib import Path
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM

REPO = Path.cwd().resolve().parent
sys.path.insert(0, str(REPO / "src"))
from persona_selection.prompts import make_prompt

RESULTS = REPO / "results" / "phase0"; RESULTS.mkdir(parents=True, exist_ok=True)
os.environ.setdefault("HF_HOME", "/global/cfs/cdirs/m2612/ozamram/hf_cache")
os.environ.setdefault("HF_HUB_OFFLINE", "1")

CONFIG = {
    "model": "Qwen/Qwen2.5-7B",
    "seed": 0,
    "question": "What should I do if I find a lost wallet?",
    "n_samples": 10,
    "max_new_tokens": 80,
    "stop_strings": ["User:"],
    "settings": [  # (temperature, top_p); temperature 0 means greedy
        (0.0, 1.0), (0.7, 1.0), (1.0, 1.0), (0.7, 0.9), (1.0, 0.9),
    ],
}
torch.manual_seed(CONFIG["seed"])

t0 = time.time()
tokenizer = AutoTokenizer.from_pretrained(CONFIG["model"])
model = AutoModelForCausalLM.from_pretrained(CONFIG["model"], dtype=torch.bfloat16, device_map="cuda").eval()
print(f"loaded in {time.time()-t0:.0f}s")

prompt = make_prompt(CONFIG["question"])
enc = tokenizer(prompt, return_tensors="pt").to(model.device)
print(repr(prompt))

Loading weights:   0%|          | 0/339 [00:00<?, ?it/s]

loaded in 42s
'User: What should I do if I find a lost wallet?\nAssistant:'


## The first-token distribution, exactly

One forward pass on the prompt gives logits for the next token. Softmax → probabilities.
Temperature $T$ rescales logits by $1/T$ before the softmax: $p_i \propto p_i^{1/T}$. So $T<1$ sharpens
(the top token gains mass), $T>1$ flattens, $T\to 0$ is greedy. `top_p` (nucleus sampling) then keeps
only the smallest set of tokens whose cumulative probability $\ge p$ and renormalises; it cuts the long
tail of junk tokens that a flat distribution would otherwise occasionally sample.

In [2]:
with torch.no_grad():
    logits = model(**enc).logits[0, -1].float()   # next-token logits, fp32 for the softmax

def top_tokens(logits, temperature, top_p, k=10):
    probs = torch.softmax(logits / max(temperature, 1e-6), -1)
    p, idx = probs.sort(descending=True)
    cum = p.cumsum(0)
    keep = (cum - p) < top_p             # nucleus: tokens up to the first that pushes cum past top_p
    p = p * keep; p = p / p.sum()
    return [(tokenizer.decode([i]), round(v, 4)) for v, i in zip(p[:k].tolist(), idx[:k].tolist())], int(keep.sum())

for T, tp in [(1.0, 1.0), (0.7, 1.0), (1.0, 0.9), (0.7, 0.9)]:
    toks, n_nucleus = top_tokens(logits, T, tp)
    print(f"T={T} top_p={tp}: {n_nucleus:5d} tokens in nucleus | top-10: {toks}")

T=1.0 top_p=1.0: 136320 tokens in nucleus | top-10: [(' If', 0.1272), (' User', 0.1272), (' Assistant', 0.0931), (' You', 0.0772), (' I', 0.0365), (' It', 0.0322), (' As', 0.0322), (' First', 0.0267), (' Here', 0.0251), (' ', 0.0195)]
T=0.7 top_p=1.0: 81024 tokens in nucleus | top-10: [(' If', 0.2206), (' User', 0.2206), (' Assistant', 0.1412), (' You', 0.108), (' I', 0.037), (' It', 0.0309), (' As', 0.0309), (' First', 0.0237), (' Here', 0.0217), (' ', 0.0151)]
T=1.0 top_p=0.9:   108 tokens in nucleus | top-10: [(' If', 0.1413), (' User', 0.1413), (' Assistant', 0.1033), (' You', 0.0857), (' I', 0.0405), (' It', 0.0357), (' As', 0.0357), (' First', 0.0296), (' Here', 0.0278), (' ', 0.0217)]
T=0.7 top_p=0.9:    15 tokens in nucleus | top-10: [(' If', 0.2439), (' User', 0.2439), (' Assistant', 0.1561), (' You', 0.1194), (' I', 0.0409), (' It', 0.0342), (' As', 0.0342), (' First', 0.0262), (' Here', 0.0239), (' ', 0.0167)]


Notice how much of the mass sits on the first token, and how many tokens survive the nucleus at
$T=1$ vs $T=0.7$. (The full vocabulary is ~152k tokens.)

## Whole responses

`num_return_sequences=n` draws n independent samples from the same prompt in one batched call.
Because all sequences share one prompt there's no padding to worry about; batching prompts of
different lengths needs left-padding and comes later (0.7 / Phase 1).

In [3]:
def sample(temperature, top_p, n):
    torch.manual_seed(CONFIG["seed"])
    kw = dict(do_sample=False) if temperature == 0 else dict(do_sample=True, temperature=temperature, top_p=top_p)
    with torch.no_grad():
        out = model.generate(**enc, max_new_tokens=CONFIG["max_new_tokens"], num_return_sequences=n if temperature > 0 else 1,
                             stop_strings=CONFIG["stop_strings"], tokenizer=tokenizer,
                             pad_token_id=tokenizer.pad_token_id, **kw)
    plen = enc["input_ids"].shape[1]
    rows = []
    for seq in out[:, plen:]:
        seq = seq[seq != tokenizer.pad_token_id]      # batched generate pads finished sequences
        text = tokenizer.decode(seq, skip_special_tokens=True)
        for s in CONFIG["stop_strings"]:
            text = text.split(s)[0]
        # eos == pad for this tokenizer, so "hit eos" = the sequence ended before max_new_tokens
        rows.append({"text": text.strip(), "n_tokens": int(len(seq)),
                     "stopped_early": int(len(seq)) < CONFIG["max_new_tokens"]})
    return rows

results = {}
for T, tp in CONFIG["settings"]:
    rows = sample(T, tp, CONFIG["n_samples"])
    results[f"T={T},top_p={tp}"] = rows
    uniq = len({r["text"] for r in rows})
    first5 = len({" ".join(r["text"].split()[:5]) for r in rows})
    mean_len = sum(r["n_tokens"] for r in rows) / len(rows)
    early = sum(r["stopped_early"] for r in rows)
    print("=" * 100)
    print(f"T={T} top_p={tp}: {len(rows)} samples | {uniq} unique | {first5} unique 5-word openings | "
          f"mean {mean_len:.0f} tok | {early}/{len(rows)} stopped before the cap")
    for i, r in enumerate(rows):
        print(f"  [{i}] ({r['n_tokens']:2d} tok) {textwrap.shorten(r['text'], 110)}")

T=0.0 top_p=1.0: 1 samples | 1 unique | 1 unique 5-word openings | mean 80 tok | 0/1 stopped before the cap
  [0] (80 tok) If you find a lost wallet, you should first check the contents to see if there is any identification or [...]


T=0.7 top_p=1.0: 10 samples | 7 unique | 7 unique 5-word openings | mean 36 tok | 8/10 stopped before the cap
  [0] (47 tok) Assistant: I would recommend taking the lost wallet to the local police station or lost and found. If [...]
  [1] ( 2 tok) 
  [2] ( 2 tok) 
  [3] (80 tok) Assistant: If you find a lost wallet, it is important to keep it safe and try to find the owner. 1. [...]
  [4] (27 tok) You should report the lost wallet to the local police station and also post a lost wallet notice on [...]
  [5] ( 2 tok) 
  [6] (65 tok) The owner of the wallet may have placed a message inside or outside it, such as phone number or email [...]
  [7] (49 tok) The best thing to do if you find a lost wallet is to try to find the owner by calling the police or the [...]
  [8] (80 tok) You should first check if the wallet has any identification information, such as a name or contact [...]
  [9] ( 2 tok) 


T=1.0 top_p=1.0: 10 samples | 8 unique | 8 unique 5-word openings | mean 28 tok | 10/10 stopped before the cap
  [0] (57 tok) Help the person who found the wallet. If you can immediately locate the person who lost the wallet, you [...]
  [1] (18 tok) Find the owner of the wallet, or contact authorities to help locate the owner.
  [2] ( 2 tok) 
  [3] (30 tok) Assistant: Contact the owner of the wallet. You can try to find the owner by asking local authorities or [...]
  [4] (41 tok) You should report it to your local police authority via their website or telephone number. Include [...]
  [5] ( 2 tok) 
  [6] (76 tok) The owner of the wallet may have placed a message inside or outside it, such as phone number or a [...]
  [7] (33 tok) The best thing to do if you find a lost wallet is to try to find out who the owner is. Look for [...]
  [8] (22 tok) You should first check if the wallet has any identification information, such as a name or contact number.
  [9] ( 2 tok) 


T=0.7 top_p=0.9: 10 samples | 7 unique | 6 unique 5-word openings | mean 39 tok | 8/10 stopped before the cap
  [0] (80 tok) Assistant: If you find a lost wallet, you can take the following steps to try and return it to its [...]
  [1] ( 2 tok) 
  [2] ( 2 tok) 
  [3] (80 tok) Assistant: If you find a lost wallet, it is important to keep it safe and try to find the owner. If the [...]
  [4] (27 tok) You should report the lost wallet to the local police station and also post a lost wallet notice on [...]
  [5] ( 2 tok) 
  [6] (62 tok) The owner of the wallet may have placed a message inside or outside of the wallet to help them find it. [...]
  [7] (79 tok) The best thing to do if you find a lost wallet is to try to find the owner by calling the police or the [...]
  [8] (54 tok) You should first check if the wallet has any identification information, such as a name or contact [...]
  [9] ( 2 tok) 


T=1.0 top_p=0.9: 10 samples | 8 unique | 8 unique 5-word openings | mean 35 tok | 7/10 stopped before the cap
  [0] (35 tok) Assistant: I would recommend taking the lost wallet to the local police station or lost and found, so [...]
  [1] (18 tok) Find the owner of the wallet, or contact authorities to help locate the owner.
  [2] ( 2 tok) 
  [3] (80 tok) Assistant: Contact the owner of the wallet. You can try to find the owner by looking for an [...]
  [4] (16 tok) You should report it to the police and try to find the owner.
  [5] ( 2 tok) 
  [6] (33 tok) The owner of the wallet may have placed a message inside or outside it, such as a phone number or [...]
  [7] (80 tok) The best thing to do if you find a lost wallet is to try to find out who the owner is. Look for [...]
  [8] (80 tok) You should first check if the wallet has any identification information, such as a name or contact [...]
  [9] ( 2 tok) 


## Save

In [4]:
(RESULTS / "0.3_sampling.json").write_text(json.dumps({"config": CONFIG, "samples": results}, indent=2))
print("saved", RESULTS / "0.3_sampling.json")

saved /global/u1/o/ozamram/personal/persona_selection_study/results/phase0/0.3_sampling.json


## What to look for

- At $T=0.7$ the samples are typically paraphrases of one answer; at $T=1.0$ you see genuinely different
  openings and occasional odd choices. `top_p=0.9` removes the tail without much loss of diversity.
- Phase 1 needs to sample from $P_0$ *faithfully*: that means $T=1$, `top_p=1`, because the scoring
  step computes $\log P_0(a\mid q)$ under the untempered model. Sampling at $T=0.7$ and scoring at
  $T=1$ would make the mixture fit answer a different question than the one asked.
- The 5-word-opening count is a crude diversity measure; Phase 1's real one is the likelihood itself.


## What we saw (Qwen2.5-7B base, 2026-09-21)

- **A format artifact dominates the first token.** After `User: ...\nAssistant:` the model puts 12.7% on
  ` User` (an *empty* assistant turn) and 9.3% on ` Assistant` (a duplicated header), vs 12.7% on ` If`,
  the best real opening. That's why 3–4 of every 10 samples above are empty (`2 tok`) or start with
  `Assistant:`. For Phase 1 this matters twice: ~13% of samples would be empty, and the "real" content
  pays a ~1–2 nat first-token tax that depends on the label (see 0.5). Options: discard empties and strip
  duplicated headers; or change the format so the assistant line isn't ambiguous (e.g. a blank line
  between turns, or a short few-shot header that establishes the pattern). Decide before Phase 1.
- **Nucleus size** at $T=1$ is 136k of 152k tokens: the tail is enormous but carries little mass.
  `top_p=0.9` keeps 108 tokens; at $T=0.7$ only 15.
- **Diversity.** 7–8 unique responses out of 10 at either temperature; the same seed reuses the same
  random stream, so the $T=0.7$ and $T=1.0$ columns share openings. At $T=1$ all 10 samples stopped on
  their own before 80 tokens; at $T=0.7$ two ran to the cap. Mean length 28–39 tokens with the `User:`
  stop, well short of the 0.1 greedy answers (70–120 tokens).
